In [ ]:
import json
import pandas as pd
from pathlib import Path
import os

os.chdir("/home/a67606/Thesis/EXIST_MEMES")

DATA = Path("./Dataset/memes/training/EXIST2026_training.json")

dataset = pd.read_json(DATA).transpose()

print(f"itens: {len(dataset)}")

dataset.head()

In [ ]:
votos = dataset[["id_EXIST", "lang", "annotators", "labels_task2_1"]].explode(
    ["annotators", "labels_task2_1"]
)

votos = votos.rename(columns={"annotators": "annotator", "labels_task2_1": "label_a"})

print(f"votos: {len(votos)}")
print(f"anotadores distintos: {votos['annotator'].nunique()}")
print(f"\nvalores em label_a:\n{votos['label_a'].value_counts(dropna=False)}")

votos.head(8)

In [ ]:
perfil = (
    votos.assign(yes=(votos["label_a"] == "YES").astype(int))
    .groupby("annotator")
    .agg(n_votes=("yes", "size"), n_yes=("yes", "sum"))
)

perfil["yes_rate"] = perfil["n_yes"] / perfil["n_votes"]

print(perfil["yes_rate"].describe())
print(f"\nmédia global de YES: {(votos['label_a'] == 'YES').mean():.4f}")
print(f"n_votes: min {perfil['n_votes'].min()}, máx {perfil['n_votes'].max()}")

perfil.sort_values("yes_rate").head(5)

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
p_global = (votos["label_a"] == "YES").mean()

# se todos votassem com a mesma probabilidade p_global,
# quanta dispersão haveria só por causa do n de cada um?
sim = rng.binomial(perfil["n_votes"].values, p_global) / perfil["n_votes"].values

print(f"sd observado : {perfil['yes_rate'].std():.4f}")
print(f"sd simulado  : {sim.std():.4f}")
print(f"razão        : {perfil['yes_rate'].std() / sim.std():.2f}x")

for lim in [0.90, 0.10]:
    obs = (perfil["yes_rate"] > lim).sum() if lim > 0.5 else (perfil["yes_rate"] < lim).sum()
    esp = (sim > lim).sum() if lim > 0.5 else (sim < lim).sum()
    print(f"\n{'>' if lim > 0.5 else '<'} {lim}: observado {obs}, esperado por acaso {esp}")

In [ ]:
perfil = (
    votos.assign(yes=(votos["label_a"] == "YES").astype(int))
    .groupby("annotator")
    .agg(n_votes=("yes", "size"), n_yes=("yes", "sum"))
)

perfil["yes_rate"] = perfil["n_yes"] / perfil["n_votes"]

print(perfil["yes_rate"].describe())
print(f"\nmédia global de YES: {(votos['label_a'] == 'YES').mean():.4f}")
print(f"n_votes: min {perfil['n_votes'].min()}, máx {perfil['n_votes'].max()}")

perfil.sort_values("yes_rate").head(5)

In [ ]:
perfil["extremo"] = (perfil["yes_rate"] > 0.90) | (perfil["yes_rate"] < 0.10)
perfil["lado"] = np.where(
    perfil["yes_rate"] > 0.90, "alto",
    np.where(perfil["yes_rate"] < 0.10, "baixo", "-")
)

print(f"extremos: {perfil['extremo'].sum()}")
print(perfil.loc[perfil["extremo"], "lado"].value_counts())

# a que língua pertencem?
lang_por_anot = votos.groupby("annotator")["lang"].agg(
    lambda s: s.iloc[0] if s.nunique() == 1 else "ambas"
)
perfil["lang"] = lang_por_anot

print("\ndistribuição por língua:")
print(pd.crosstab(perfil["lang"], perfil["lado"], margins=True))

print("\nn_votes dos extremos:")
print(perfil.loc[perfil["extremo"], "n_votes"].describe())

In [ ]:
# taxa de YES do pool, por língua
print(votos.groupby("lang")["label_a"].apply(lambda s: (s == "YES").mean()))

# para cada anotador, a taxa média de YES dos OUTROS 5 nos itens que ele viu
soft = votos.assign(yes=(votos["label_a"] == "YES").astype(int))
por_item = soft.groupby("id_EXIST")["yes"].agg(["sum", "size"])

soft = soft.join(por_item, on="id_EXIST")
soft["outros"] = (soft["sum"] - soft["yes"]) / (soft["size"] - 1)

perfil["baseline_itens"] = soft.groupby("annotator")["outros"].mean()
perfil["desvio"] = perfil["yes_rate"] - perfil["baseline_itens"]

print("\ndesvio face aos itens que viu, por lado e língua:")
print(perfil.groupby(["lado", "lang"])["desvio"].agg(["mean", "count"]))

In [ ]:
def soft_labels(votos_df):
    g = votos_df.assign(yes=(votos_df["label_a"] == "YES").astype(int)).groupby("id_EXIST")
    return g["yes"].mean(), g["yes"].size()

base, n_base = soft_labels(votos)

for nome, remover in [
    ("alto (40)",  perfil.index[perfil["lado"] == "alto"]),
    ("baixo (21)", perfil.index[perfil["lado"] == "baixo"]),
    ("ambos (61)", perfil.index[perfil["extremo"]]),
]:
    sub = votos[~votos["annotator"].isin(remover)]
    lab, n = soft_labels(sub)
    d = (lab - base).abs()
    print(f"\n{nome}")
    print(f"  itens afetados     : {(d > 1e-9).sum()} de {len(base)}")
    print(f"  média |Δ| (afetados): {d[d > 1e-9].mean():.4f}")
    print(f"  máx |Δ|            : {d.max():.4f}")
    print(f"  cruzam 0,5         : {((base < 0.5) != (lab < 0.5)).sum()}")
    print(f"  itens com <5 votos : {(n < 5).sum()}, com <3: {(n < 3).sum()}")

In [ ]:
def sortear_controle(seed):
    rng_local = np.random.default_rng(seed)
    alvo = perfil.loc[perfil["extremo"], ["lang", "n_votes"]]
    pool = perfil.loc[~perfil["extremo"]]
    escolhidos = []

    for (lg, nv), grupo in alvo.groupby(["lang", "n_votes"]):
        cand = pool.index[
            (pool["lang"] == lg)
            & (pool["n_votes"] == nv)
            & (~pool.index.isin(escolhidos))
        ].to_numpy()
        k = len(grupo)
        if len(cand) < k:  # estrato exato insuficiente: relaxa n_votes, mantém língua
            cand = pool.index[
                (pool["lang"] == lg)
                & (pool["n_votes"].between(nv - 3, nv + 3))
                & (~pool.index.isin(escolhidos))
            ].to_numpy()
        escolhidos.extend(rng_local.choice(cand, size=k, replace=False))

    return pd.Index(escolhidos)

for seed in [42, 7, 2025]:
    ctrl = sortear_controle(seed)
    sub = votos[~votos["annotator"].isin(ctrl)]
    lab, n = soft_labels(sub)
    d = (lab - base).abs()
    print(f"\ncontrole seed {seed}  (n={len(ctrl)})")
    print(f"  língua      : {perfil.loc[ctrl, 'lang'].value_counts().to_dict()}")
    print(f"  yes_rate    : média {perfil.loc[ctrl, 'yes_rate'].mean():.3f}, "
          f"min {perfil.loc[ctrl, 'yes_rate'].min():.3f}, "
          f"máx {perfil.loc[ctrl, 'yes_rate'].max():.3f}")
    print(f"  itens afetados      : {(d > 1e-9).sum()}")
    print(f"  média |Δ| (afetados): {d[d > 1e-9].mean():.4f}")
    print(f"  cruzam 0,5          : {((base < 0.5) != (lab < 0.5)).sum()}")
    print(f"  itens com <5 votos  : {(n < 5).sum()}")

In [ ]:
MAJ = 0.5  # empate impossível com 5 votos, mas explícito

def substituir(votos_df, alvos):
    """Troca o voto dos anotadores em `alvos` pelo majoritário dos outros do item."""
    v = votos_df.assign(yes=(votos_df["label_a"] == "YES").astype(int)).copy()
    tot = v.groupby("id_EXIST")["yes"].transform("sum")
    n = v.groupby("id_EXIST")["yes"].transform("size")
    outros = (tot - v["yes"]) / (n - 1)
    alvo = v["annotator"].isin(alvos)
    v.loc[alvo, "yes"] = (outros[alvo] > MAJ).astype(int)
    return v

def labels_de(v):
    g = v.groupby("id_EXIST")["yes"]
    return g.mean(), g.size()

votos_bin = votos.assign(yes=(votos["label_a"] == "YES").astype(int))
extremos = perfil.index[perfil["extremo"]]
ctrl42 = sortear_controle(42)

bracos = {
    "A0_baseline":    votos_bin,
    "R1_rem_extremos": votos_bin[~votos_bin["annotator"].isin(extremos)],
    "R2_rem_controle": votos_bin[~votos_bin["annotator"].isin(ctrl42)],
    "S1_sub_extremos": substituir(votos, extremos),
    "S2_sub_controle": substituir(votos, ctrl42),
}

base_lab, _ = labels_de(votos_bin)
for nome, v in bracos.items():
    lab, n = labels_de(v)
    d = (lab - base_lab).abs()
    print(f"{nome:18s} afet={(d>1e-9).sum():5d}  "
          f"|Δ|={d[d>1e-9].mean() if (d>1e-9).any() else 0:.4f}  "
          f"cruz={((base_lab<0.5)!=(lab<0.5)).sum():4d}  "
          f"n<5={(n<5).sum():4d}")

In [ ]:
long_bc = dataset[["id_EXIST", "lang", "annotators",
                   "labels_task2_1", "labels_task2_2", "labels_task2_3"]].explode(
    ["annotators", "labels_task2_1", "labels_task2_2", "labels_task2_3"]
).rename(columns={"annotators": "annotator",
                  "labels_task2_1": "label_a",
                  "labels_task2_2": "label_b",
                  "labels_task2_3": "label_c"})

sexistas = long_bc[long_bc["label_a"] == "YES"]

dens = sexistas.groupby("annotator").size().rename("n_sexist")
perfil["n_sexist"] = dens.reindex(perfil.index).fillna(0).astype(int)

print(perfil.groupby("lado")["n_sexist"].describe()[["count", "mean", "50%", "min", "max"]])

print("\nintenção (B) por lado:")
print(pd.crosstab(perfil["lado"].reindex(sexistas["annotator"]).values,
                  sexistas["label_b"].values, normalize="index").round(3))

In [ ]:
alt = perfil.index[perfil["lado"] == "alto"]
sex = sexistas.copy()
sex["grupo"] = np.where(sex["annotator"].isin(alt), "alto", "resto")

# quantas categorias por voto, e quais
sex["n_cat"] = sex["label_c"].apply(lambda x: len([c for c in x if c != "-"]))
print(sex.groupby("grupo")["n_cat"].agg(["mean", "median", "count"]))

from collections import Counter
for g, sub in sex.groupby("grupo"):
    c = Counter(cat for lst in sub["label_c"] for cat in lst if cat != "-")
    tot = sum(c.values())
    print(f"\n{g}:", {k: round(v/tot, 3) for k, v in c.most_common()})

In [ ]:
from pathlib import Path
OUT = Path("./reliability/outputs/ablation")
OUT.mkdir(parents=True, exist_ok=True)

def escrever(nome, v):
    lab, n = labels_de(v)
    payload = {str(k): [float(lab[k]), float(1.0 - lab[k])] for k in lab.index}
    p = OUT / f"soft_labels_taskA_{nome}.json"
    with open(p, "w") as f:
        json.dump(payload, f, indent=1)
    print(f"{nome:16s} {len(payload)} itens  ->  {p}")

escrever("A0_baseline",     bracos["A0_baseline"])
escrever("R1_rem_extremos", bracos["R1_rem_extremos"])
escrever("R2_rem_controle", bracos["R2_rem_controle"])

# verificação contra o ficheiro que o pipeline já consome
novo = json.load(open(OUT / "soft_labels_taskA_A0_baseline.json"))
raw_ref = json.load(open("../reliability/outputs/soft_labels_taskA_raw_fixed.json"))
difs = [(k, novo[k], raw_ref[k]) for k in raw_ref
        if abs(novo[k][0] - raw_ref[k][0]) > 1e-9]
print(f"\nchaves em comum: {len(set(novo) & set(raw_ref))} de {len(raw_ref)}")
print(f"divergências A0 vs raw: {len(difs)}")
if difs:
    print(difs[:3])

In [ ]:
raw     = json.load(open("../reliability/outputs/soft_labels_taskA_raw.json"))
raw_fix = json.load(open("../reliability/outputs/soft_labels_taskA_raw_fixed.json"))

d = [k for k in raw if abs(raw[k][0] - raw_fix[k][0]) > 1e-9]
print(f"raw vs raw_fixed: {len(d)} divergências, chaves {len(raw)} vs {len(raw_fix)}")

for nome, ref in [("raw", raw), ("raw_fixed", raw_fix)]:
    difs = [k for k in ref if abs(novo[k][0] - ref[k][0]) > 1e-9]
    print(f"A0 vs {nome:10s}: comuns {len(set(novo) & set(ref))}/{len(ref)}, "
          f"divergências {len(difs)}")
    if difs:
        print("   ex:", [(k, novo[k], ref[k]) for k in difs[:3]])

In [ ]:
import json
existente = json.load(open("../reliability/outputs/soft_labels_taskA_weighted.json"))
novo = json.load(open("./reliability/outputs/ablation/soft_labels_taskA_A0_baseline.json"))

k = next(iter(existente))
print("existente:", k, "->", existente[k])
print("novo     :", k, "->", novo.get(k))

In [ ]:
import numpy as np

def resumo(v, nome):
    lab, n = labels_de(v)
    print(f"{nome:18s} média {lab.mean():.4f}  sd {lab.std():.4f}  "
          f"P(>0,5) {(lab > 0.5).mean():.4f}  "
          f"|0| {(lab == 0).sum():4d}  |1| {(lab == 1).sum():4d}  "
          f"valores {lab.nunique():2d}")

for nome, v in bracos.items():
    resumo(v, nome)

# histograma comparado
print("\nhistograma (10 bins):")
h = {}
for nome in ["A0_baseline", "R1_rem_extremos", "R2_rem_controle"]:
    lab, _ = labels_de(bracos[nome])
    h[nome] = np.histogram(lab, bins=10, range=(0, 1))[0]
print(pd.DataFrame(h, index=[f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)]))

In [ ]:
import json, hashlib, os, time

p = "../reliability/outputs/split_taskA_frozen.json"
s = json.load(open(p))
print(f"train {len(s['train_ids'])}  val {len(s['val_ids'])}  total {len(s['train_ids'])+len(s['val_ids'])}")
print("sha256:", hashlib.sha256(open(p,'rb').read()).hexdigest()[:16])
print("modificado:", time.ctime(os.path.getmtime(p)))

In [ ]:
import json, glob
for p in sorted(glob.glob("./metrics_history/*raw*.json")):
    d = json.load(open(p))
    n = len(d) if isinstance(d, list) else len(d.get("history", d))
    print(f"{p.split('/')[-1]:40s} avaliações: {n}")